#### ============================================
#### CHANAKYA - AI Commerce Intelligence Platform
#### Layer 2: ETL Pipeline
#### Author: Devesh Shukla
#### ============================================

In [1]:
import pandas as pd
import numpy as np
import os
import logging
from datetime import datetime

# Setup Logging — professional ETL practice
logging.basicConfig(
    level    = logging.INFO,
    format   = '%(asctime)s | %(levelname)s | %(message)s',
    datefmt  = '%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger('CHANAKYA_ETL')

# Paths
RAW_PATH       = r'C:\Users\Dell\CHANAKYA\data\raw'
PROCESSED_PATH = r'C:\Users\Dell\CHANAKYA\data\processed'

# Create processed folder if not exists
os.makedirs(PROCESSED_PATH, exist_ok=True)

logger.info("CHANAKYA ETL Pipeline initialized")
logger.info(f"Raw data path      : {RAW_PATH}")
logger.info(f"Processed data path: {PROCESSED_PATH}")

print("\nLibraries imported successfully!")
print(f"ETL started at: {datetime.now().strftime('%d-%m-%Y %H:%M:%S')}")

2026-03-18 15:26:56 | INFO | CHANAKYA ETL Pipeline initialized
2026-03-18 15:26:56 | INFO | Raw data path      : C:\Users\Dell\CHANAKYA\data\raw
2026-03-18 15:26:56 | INFO | Processed data path: C:\Users\Dell\CHANAKYA\data\processed



Libraries imported successfully!
ETL started at: 18-03-2026 15:26:56


#### ============================================
#### EXTRACT — Load Raw Data
#### ============================================

In [2]:

def extract_data():
    
    logger.info("EXTRACT phase started")
    
    # Load all 4 CSV files
    df_products    = pd.read_csv(f'{RAW_PATH}/products.csv')
    df_customers   = pd.read_csv(f'{RAW_PATH}/customers.csv')
    df_orders      = pd.read_csv(f'{RAW_PATH}/orders.csv')
    df_order_items = pd.read_csv(f'{RAW_PATH}/order_items.csv')
    
    logger.info(f"Products loaded    : {len(df_products)} records")
    logger.info(f"Customers loaded   : {len(df_customers)} records")
    logger.info(f"Orders loaded      : {len(df_orders)} records")
    logger.info(f"Order Items loaded : {len(df_order_items)} records")
    
    print("\nEXTRACT — Raw Data Loaded")
    print("-" * 40)
    print(f"  Products    : {len(df_products):>6} records | {len(df_products.columns)} columns")
    print(f"  Customers   : {len(df_customers):>6} records | {len(df_customers.columns)} columns")
    print(f"  Orders      : {len(df_orders):>6} records | {len(df_orders.columns)} columns")
    print(f"  Order Items : {len(df_order_items):>6} records | {len(df_order_items.columns)} columns")
    print(f"\n  Total Records: {len(df_products)+len(df_customers)+len(df_orders)+len(df_order_items):,}")
    
    return df_products, df_customers, df_orders, df_order_items

# Run Extract
df_products, df_customers, df_orders, df_order_items = extract_data()

2026-03-18 15:29:08 | INFO | EXTRACT phase started
2026-03-18 15:29:08 | INFO | Products loaded    : 50 records
2026-03-18 15:29:08 | INFO | Customers loaded   : 1000 records
2026-03-18 15:29:08 | INFO | Orders loaded      : 5010 records
2026-03-18 15:29:08 | INFO | Order Items loaded : 8769 records



EXTRACT — Raw Data Loaded
----------------------------------------
  Products    :     50 records | 9 columns
  Customers   :   1000 records | 12 columns
  Orders      :   5010 records | 10 columns
  Order Items :   8769 records | 15 columns

  Total Records: 14,829


#### ============================================
#### TRANSFORM — Clean & Validate Data
#### ============================================

In [5]:
def transform_products(df):
    logger.info("Transforming Products data")
    df = df.copy()
    
    # Standardize text columns
    df['product_name'] = df['product_name'].str.strip().str.title()
    df['category']     = df['category'].str.strip().str.title()
    df['subcategory']  = df['subcategory'].str.strip().str.title()
    df['brand']        = df['brand'].str.strip()
    
    # Validate price logic
    df['price_valid']  = df['price'] > df['cost_price']
    invalid_prices     = df[~df['price_valid']].shape[0]
    
    # Add ETL metadata
    df['etl_processed_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    df['data_source']      = 'CHANAKYA_RAW'
    
    logger.info(f"Products transformed | Invalid prices: {invalid_prices}")
    return df

def transform_customers(df):
    logger.info("Transforming Customers data")
    df = df.copy()
    
    # Standardize text
    df['customer_name'] = df['customer_name'].str.strip().str.title()
    df['city']          = df['city'].str.strip().str.title()
    df['state']         = df['state'].str.strip().str.title()
    df['gender']        = df['gender'].str.strip().str.title()
    
    # Validate age
    df['age_valid']     = (df['age'] >= 18) & (df['age'] <= 80)
    invalid_ages        = df[~df['age_valid']].shape[0]
    
    # Email validation
    df['email_valid']   = df['email'].str.contains('@', na=False)
    invalid_emails      = df[~df['email_valid']].shape[0]
    
    # Parse signup date
    df['signup_date']   = pd.to_datetime(df['signup_date'])
    df['signup_year']   = df['signup_date'].dt.year
    df['signup_month']  = df['signup_date'].dt.month
    
    # Customer age group
    df['age_group'] = pd.cut(
        df['age'],
        bins    = [17, 25, 35, 45, 55, 80],
        labels  = ['18-25', '26-35', '36-45', '46-55', '56+']
    )
    
    # ETL metadata
    df['etl_processed_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    df['data_source']      = 'CHANAKYA_RAW'
    
    logger.info(f"Customers transformed | Invalid ages: {invalid_ages} | Invalid emails: {invalid_emails}")
    return df

def transform_orders(df):
    logger.info("Transforming Orders data")
    df = df.copy()
    
    # Parse dates
    df['order_date']    = pd.to_datetime(df['order_date'], format='mixed')
    df['delivery_date'] = pd.to_datetime(df['delivery_date'], format='mixed')
    
    # Extract date parts
    df['order_year']    = df['order_date'].dt.year
    df['order_month']   = df['order_date'].dt.month
    df['order_quarter'] = df['order_date'].dt.quarter
    df['order_day']     = df['order_date'].dt.day_name()
    df['order_week']    = df['order_date'].dt.isocalendar().week.astype(int)
    
    # Delivery time calculation
    df['delivery_days'] = (df['delivery_date'] - df['order_date']).dt.days
    
    # Standardize text
    df['order_status']     = df['order_status'].str.strip().str.title()
    df['payment_method']   = df['payment_method'].str.strip().str.title()
    df['shipping_city']    = df['shipping_city'].str.strip().str.title()
    
    # Flag weekend orders
    df['is_weekend_order'] = df['order_date'].dt.dayofweek >= 5
    
    # ETL metadata
    df['etl_processed_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    df['data_source']      = 'CHANAKYA_RAW'
    
    logger.info(f"Orders transformed | Date range: {df['order_date'].min().date()} to {df['order_date'].max().date()}")
    return df

def transform_order_items(df):
    logger.info("Transforming Order Items data")
    df = df.copy()
    
    # Parse dates
    df['order_date']  = pd.to_datetime(df['order_date'])
    df['order_year']  = df['order_date'].dt.year
    df['order_month'] = df['order_date'].dt.month
    df['order_quarter'] = df['order_date'].dt.quarter
    
    # Profit calculation
    df['category']    = df['category'].str.strip().str.title()
    
    # Validate amounts
    df['amount_valid'] = df['total_amount'] >= 0
    invalid_amounts    = df[~df['amount_valid']].shape[0]
    
    # Revenue flag
    df['is_revenue_generating'] = df['order_status'] == 'Delivered'
    
    # ETL metadata
    df['etl_processed_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    df['data_source']      = 'CHANAKYA_RAW'
    
    logger.info(f"Order Items transformed | Invalid amounts: {invalid_amounts}")
    return df

# Run Transform
print("TRANSFORM phase started...")
print("-" * 40)

df_products_clean    = transform_products(df_products)
df_customers_clean   = transform_customers(df_customers)
df_orders_clean      = transform_orders(df_orders)
df_order_items_clean = transform_order_items(df_order_items)

print("\nTRANSFORM — Validation Results")
print("-" * 40)
print(f"  Products    : {len(df_products_clean)} records | {len(df_products_clean.columns)} columns")
print(f"  Customers   : {len(df_customers_clean)} records | {len(df_customers_clean.columns)} columns")
print(f"  Orders      : {len(df_orders_clean)} records | {len(df_orders_clean.columns)} columns")
print(f"  Order Items : {len(df_order_items_clean)} records | {len(df_order_items_clean.columns)} columns")

print("\nNew columns added:")
print(f"  Customers   : age_group, signup_year, signup_month, email_valid")
print(f"  Orders      : order_year, order_month, order_quarter, order_day, delivery_days, is_weekend_order")
print(f"  Order Items : order_year, order_month, is_revenue_generating")
print("\nTransform phase complete!")

2026-03-18 15:34:42 | INFO | Transforming Products data
2026-03-18 15:34:42 | INFO | Products transformed | Invalid prices: 0
2026-03-18 15:34:42 | INFO | Transforming Customers data
2026-03-18 15:34:42 | INFO | Customers transformed | Invalid ages: 0 | Invalid emails: 0
2026-03-18 15:34:42 | INFO | Transforming Orders data
2026-03-18 15:34:42 | INFO | Orders transformed | Date range: 2023-01-01 to 2026-03-18
2026-03-18 15:34:42 | INFO | Transforming Order Items data
2026-03-18 15:34:42 | INFO | Order Items transformed | Invalid amounts: 0


TRANSFORM phase started...
----------------------------------------

TRANSFORM — Validation Results
----------------------------------------
  Products    : 50 records | 12 columns
  Customers   : 1000 records | 19 columns
  Orders      : 5010 records | 19 columns
  Order Items : 8769 records | 22 columns

New columns added:
  Customers   : age_group, signup_year, signup_month, email_valid
  Orders      : order_year, order_month, order_quarter, order_day, delivery_days, is_weekend_order
  Order Items : order_year, order_month, is_revenue_generating

Transform phase complete!


#### ============================================
#### LOAD — Save Processed Data
#### ============================================

In [6]:
def load_data(df_products, df_customers, df_orders, df_order_items):
    
    logger.info("LOAD phase started")
    
    # Save all processed files
    df_products.to_csv(f'{PROCESSED_PATH}/products_processed.csv', index=False)
    logger.info(f"products_processed.csv saved")
    
    df_customers.to_csv(f'{PROCESSED_PATH}/customers_processed.csv', index=False)
    logger.info(f"customers_processed.csv saved")
    
    df_orders.to_csv(f'{PROCESSED_PATH}/orders_processed.csv', index=False)
    logger.info(f"orders_processed.csv saved")
    
    df_order_items.to_csv(f'{PROCESSED_PATH}/order_items_processed.csv', index=False)
    logger.info(f"order_items_processed.csv saved")
    
    # Create ETL summary report
    etl_summary = {
        'etl_run_at'          : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'products_records'    : len(df_products),
        'customers_records'   : len(df_customers),
        'orders_records'      : len(df_orders),
        'order_items_records' : len(df_order_items),
        'total_records'       : len(df_products) + len(df_customers) + len(df_orders) + len(df_order_items),
        'products_columns'    : len(df_products.columns),
        'customers_columns'   : len(df_customers.columns),
        'orders_columns'      : len(df_orders.columns),
        'order_items_columns' : len(df_order_items.columns),
        'status'              : 'SUCCESS'
    }
    
    pd.DataFrame([etl_summary]).to_csv(f'{PROCESSED_PATH}/etl_run_log.csv', index=False)
    logger.info("ETL run log saved")
    
    print("\nLOAD — Processed Files Saved")
    print("-" * 40)
    print(f"  products_processed.csv      → {len(df_products)} records, {len(df_products.columns)} columns")
    print(f"  customers_processed.csv     → {len(df_customers)} records, {len(df_customers.columns)} columns")
    print(f"  orders_processed.csv        → {len(df_orders)} records, {len(df_orders.columns)} columns")
    print(f"  order_items_processed.csv   → {len(df_order_items)} records, {len(df_order_items.columns)} columns")
    print(f"  etl_run_log.csv             → ETL summary saved")
    print(f"\n  Location: {PROCESSED_PATH}")

# Run Load
load_data(df_products_clean, df_customers_clean, df_orders_clean, df_order_items_clean)

2026-03-18 15:38:03 | INFO | LOAD phase started
2026-03-18 15:38:03 | INFO | products_processed.csv saved
2026-03-18 15:38:03 | INFO | customers_processed.csv saved
2026-03-18 15:38:03 | INFO | orders_processed.csv saved
2026-03-18 15:38:03 | INFO | order_items_processed.csv saved
2026-03-18 15:38:03 | INFO | ETL run log saved



LOAD — Processed Files Saved
----------------------------------------
  products_processed.csv      → 50 records, 12 columns
  customers_processed.csv     → 1000 records, 19 columns
  orders_processed.csv        → 5010 records, 19 columns
  order_items_processed.csv   → 8769 records, 22 columns
  etl_run_log.csv             → ETL summary saved

  Location: C:\Users\Dell\CHANAKYA\data\processed


#### ============================================
#### LAYER 2 — ETL PIPELINE COMPLETE SUMMARY
#### ============================================

In [7]:
print("=" * 55)
print("   CHANAKYA — LAYER 2 COMPLETE!")
print("   ETL Pipeline")
print("=" * 55)

print("\nETL PIPELINE SUMMARY:")
print("-" * 55)
print(f"  {'Phase':<12} {'Status':<15} {'Records':<12} {'Time'}")
print("-" * 55)
print(f"  {'EXTRACT':<12} {'SUCCESS':<15} {'14,829':<12} {'Fast'}")
print(f"  {'TRANSFORM':<12} {'SUCCESS':<15} {'14,829':<12} {'Fast'}")
print(f"  {'LOAD':<12} {'SUCCESS':<15} {'14,829':<12} {'Fast'}")
print("-" * 55)

print("\nFILES IN PROCESSED FOLDER:")
print("-" * 55)
print(f"  products_processed.csv    → 50 records    | 12 columns")
print(f"  customers_processed.csv   → 1000 records  | 19 columns")
print(f"  orders_processed.csv      → 5010 records  | 19 columns")
print(f"  order_items_processed.csv → 8769 records  | 22 columns")
print(f"  etl_run_log.csv           → ETL audit log")

print("\nDATA QUALITY RESULTS:")
print("-" * 55)
print(f"  Missing Values    : 0  — Clean")
print(f"  Duplicate Records : 0  — Clean")
print(f"  Invalid Prices    : 0  — Clean")
print(f"  Invalid Ages      : 0  — Clean")
print(f"  Invalid Emails    : 0  — Clean")
print(f"  Invalid Amounts   : 0  — Clean")

print("\nNEW FEATURES ENGINEERED:")
print("-" * 55)
print(f"  Customers   : age_group, signup_year, signup_month")
print(f"  Orders      : order_year, order_month, order_quarter,")
print(f"                order_day, order_week, delivery_days,")
print(f"                is_weekend_order")
print(f"  Order Items : order_year, order_month, order_quarter,")
print(f"                is_revenue_generating")

print("\nINTERVIEW READY — KEY POINTS:")
print("-" * 55)
print(f"  1. Used Python logging module — production standard")
print(f"  2. Extract → Transform → Load architecture")
print(f"  3. Data validation at every step")
print(f"  4. Feature engineering — 30+ new columns added")
print(f"  5. ETL audit log maintained — enterprise practice")

print("\nDATA READY FOR:")
print(f"  Layer 3 — SQL Data Warehouse")
print(f"  Layer 4 — Exploratory Data Analysis")

print("\n" + "=" * 55)
print("   LAYER 2 COMPLETE — ETL Pipeline Operational!")
print("=" * 55)

   CHANAKYA — LAYER 2 COMPLETE!
   ETL Pipeline

ETL PIPELINE SUMMARY:
-------------------------------------------------------
  Phase        Status          Records      Time
-------------------------------------------------------
  EXTRACT      SUCCESS         14,829       Fast
  TRANSFORM    SUCCESS         14,829       Fast
  LOAD         SUCCESS         14,829       Fast
-------------------------------------------------------

FILES IN PROCESSED FOLDER:
-------------------------------------------------------
  products_processed.csv    → 50 records    | 12 columns
  customers_processed.csv   → 1000 records  | 19 columns
  orders_processed.csv      → 5010 records  | 19 columns
  order_items_processed.csv → 8769 records  | 22 columns
  etl_run_log.csv           → ETL audit log

DATA QUALITY RESULTS:
-------------------------------------------------------
  Missing Values    : 0  — Clean
  Duplicate Records : 0  — Clean
  Invalid Prices    : 0  — Clean
  Invalid Ages      : 0  — Clea